# 1. Sequence Reranker: LSTM/GRU Click History

Treats the user's click history as an ordered sequence rather than a set.

- **`ArticleEncoder`**: frozen 64-dimensional text embeddings shared by candidates and history
- **`HistorySequenceEncoder`**: an LSTM or GRU reads the clicks in chronological order; the
  last valid step is the user vector, and an empty history gives a zero vector
- **Tabular branch** (optional): the frozen causal tabular features used by the MLP baseline
- **`CandidateScorer`**: an MLP over `[tabular ; candidate ; user]` that outputs one logit

Two training objectives are available:

- **`"bce_bpr"`**: $0.75\,\text{weighted BCE} + 0.25\,\text{BPR}$, with BPR pairs taken from the
  negative groups saved by feature engineering
- **`"lambda_ndcg"`**: LambdaNDCG@K, a listwise loss over each retained training impression

Set `RNN_TYPE`, `OBJECTIVE` and `USE_TABULAR` and rerun to compare variants on validation;
section 7 compares the runs with and without the tabular branch.

## 1.1 Load Packages

In [ ]:
import json
import os
import random
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import polars as pl
import pyarrow.parquet as pq
import torch
import torch.nn.functional as F
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

## 1.2 Set Up Paths

Set `ENVIRONMENT` to `"colab"` to read the project and data from Google Drive. The
repository supplies the shared evaluator; the data folder holds the prepared features
and receives every checkpoint, score file and metric table this notebook writes.

In [ ]:
ENVIRONMENT = "local"  # "local" or "colab"

if ENVIRONMENT == "colab":
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/BT4222-Group-6-")
    DATA_ROOT = Path("/content/drive/MyDrive/v3-mindsmall")
else:
    PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()
    while (
        not (PROJECT_ROOT / "5_Models").exists() and PROJECT_ROOT != PROJECT_ROOT.parent
    ):
        PROJECT_ROOT = PROJECT_ROOT.parent
    DATA_ROOT = Path(os.environ.get("DATA_ROOT", PROJECT_ROOT)).resolve()

# Ranking features, click labels and causal user histories from feature engineering
RUN = DATA_ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"

# Frozen article embeddings from the representation stage
REPRESENTATIONS = DATA_ROOT / "3_Representations" / "artifacts"

# Checkpoints, score files and metric tables written by the reranking notebooks
ARTIFACTS = DATA_ROOT / "5_Models" / "5_02_Reranking" / "artifacts"
RESULTS = DATA_ROOT / "results"

# The shared evaluator is the only project module the reranking notebooks import
sys.path.insert(0, str(PROJECT_ROOT / "5_Models" / "5_02_Reranking"))
from rr_evaluation import compare_results, rank_logged_slate, score_and_evaluate

## 1.3 Load Ranking Features

Each split is stored as Parquet partitions with one row per (impression, candidate)
pair. These previews read only the first rows; later cells load what they need.

In [ ]:
def preview_split(split, n_rows=5):
    """Read the first rows of a split without loading every partition."""
    first_part = min((RUN / f"ranking_features_{split}").glob("*.parquet"))
    return next(pq.ParquetFile(first_part).iter_batches(batch_size=n_rows)).to_pandas()


train_preview = preview_split("train")
validation_preview = preview_split("validation")
train_preview.head()

In [ ]:
validation_preview.head()

# 2. Experiment Configuration

In [ ]:
SEED = 4222
DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f"Using {DEVICE}")

RNN_TYPE = "lstm"  # "lstm" or "gru"
OBJECTIVE = "bce_bpr"  # "bce_bpr" or "lambda_ndcg"
BPR_WEIGHT = 0.25  # bce_bpr: loss = (1 - BPR_WEIGHT) * BCE + BPR_WEIGHT * BPR
NDCG_K = 5  # lambda_ndcg: cut-off of the optimised nDCG
BATCH_IMPRESSIONS = 256  # lambda_ndcg: impressions per training batch

SELECTION_METRIC = "ndcg_at_10"
HISTORY_CAP = 50
HIDDEN_DIMS, DROPOUT = (256, 64), 0.20
EPOCHS = 5
BATCH_SIZE = 1024

In [ ]:
# The frozen feature schema lists the causal tabular inputs shared by every reranker
schema = json.loads((RUN / "feature_schema.json").read_text())
FEATURES = schema["model_columns"]
CATEGORICAL = [
    name for name in schema.get("categorical_columns", []) if name in FEATURES
]
NUMERIC = [name for name in FEATURES if name not in CATEGORICAL]
print(
    f"{len(FEATURES)} tabular features: {len(NUMERIC)} numeric, {len(CATEGORICAL)} categorical"
)

In [ ]:
USE_TABULAR = True  # False = text + click history only (the branch ablation)
BRANCHES = "tabular_text" if USE_TABULAR else "text"
TABULAR_COLUMNS = FEATURES if USE_TABULAR else []
print(f"Branches: {BRANCHES}")

from experiment_naming import sequence_name

objective_name = (
    f"bce{1 - BPR_WEIGHT:.2f}_bpr{BPR_WEIGHT:.2f}"
    if OBJECTIVE == "bce_bpr"
    else f"lambda_ndcg{NDCG_K}"
)
EXPERIMENT_NAME = sequence_name(
    rnn_type=RNN_TYPE,
    objective=objective_name,
    history_cap=HISTORY_CAP,
    hidden_dims=HIDDEN_DIMS,
    dropout=DROPOUT,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    seed=SEED,
    branches=BRANCHES,
)
OUTPUT_DIR = ARTIFACTS / EXPERIMENT_NAME
print(f"Experiment: {EXPERIMENT_NAME}")

# 3. Prepare Training Data

## 3.1 Load All Retained Training Candidates

This model trains on every candidate kept by feature engineering (clicks plus the
pre-sampled unclicked candidates), not a further sample.

In [ ]:
def load_training_data():
    """Read every retained training candidate with its click label."""
    frames = []
    for part in sorted((RUN / "ranking_features_train").glob("*.parquet")):
        features = pq.read_table(
            part, columns=["impression_uid", "news_code", *TABULAR_COLUMNS]
        ).to_pandas()
        labels = pq.read_table(
            RUN / "labels_train" / part.name,
            columns=["impression_uid", "news_code", "clicked"],
        ).to_pandas()
        frames.append(
            features.merge(
                labels, on=["impression_uid", "news_code"], validate="one_to_one"
            )
        )
    return pd.concat(frames, ignore_index=True)


def load_saved_pairs(train):
    """Map the click/non-click groups saved by feature engineering to training row indices."""
    groups = pl.read_parquet(RUN / "training_negative_groups" / "*.parquet")
    pairs = (
        groups.explode("negative_news_codes", empty_as_null=True)
        .drop_nulls("negative_news_codes")
        .to_pandas()
    )

    row_index = pd.MultiIndex.from_frame(train[["impression_uid", "news_code"]])
    positive = row_index.get_indexer(
        pd.MultiIndex.from_arrays(
            [pairs["impression_uid"], pairs["positive_news_code"]]
        )
    )
    negative = row_index.get_indexer(
        pd.MultiIndex.from_arrays(
            [pairs["impression_uid"], pairs["negative_news_codes"]]
        )
    )
    if (positive < 0).any() or (negative < 0).any():
        raise ValueError("A saved BPR pair is missing from the training features")
    return positive.astype(np.int64), negative.astype(np.int64)

## 3.2 Fit Feature Preprocessing

Fitted on training rows only. With `USE_TABULAR = False` the model receives a zero-width
tabular input, so the text-only run uses the same architecture minus the tabular branch.

In [ ]:
def build_preprocessor():
    """Median-impute and standardise numeric inputs; mode-impute and one-hot encode categoricals."""
    numeric = Pipeline(
        [
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]
    )
    categorical = Pipeline(
        [
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )
    return ColumnTransformer(
        [
            ("numeric", numeric, NUMERIC),
            ("categorical", categorical, CATEGORICAL),
        ]
    )


def tabular_inputs(frame):
    """Preprocessed tabular features, or a zero-width array when the tabular branch is off."""
    if not USE_TABULAR:
        return np.zeros((len(frame), 0), dtype=np.float32)
    return preprocessor.transform(frame[FEATURES]).astype(np.float32, copy=False)

## 3.3 Article Vectors and Causal Click Histories

Histories are **right-padded** here (oldest click first) so the recurrent layer reads
them in chronological order and the last valid step is the most recent click.

In [ ]:
# Map each news_code to its row in the frozen 64-dimensional text embedding matrix.
# Row 0 is reserved for padding.
article_index = pl.read_parquet(
    REPRESENTATIONS / "article_index.parquet", columns=["news_code", "article_row"]
)
news_to_row = dict(
    zip(article_index["news_code"].to_list(), article_index["article_row"].to_list())
)
text_vectors = np.load(REPRESENTATIONS / "text64.npy", mmap_mode="r")
EMBEDDING_DIM = text_vectors.shape[1]
print(
    f"Article embeddings: {text_vectors.shape[0]:,} rows x {EMBEDDING_DIM} dimensions"
)

In [ ]:
def load_profiles(split, impression_ids=None):
    """Map each impression to the article rows its user clicked strictly before the impression."""
    profiles = pl.scan_parquet(RUN / f"request_profiles_{split}" / "*.parquet").select(
        "impression_uid", "full_history_rows"
    )
    if impression_ids is not None:
        profiles = profiles.filter(pl.col("impression_uid").is_in(list(impression_ids)))
    profiles = profiles.collect()
    return dict(
        zip(
            profiles["impression_uid"].to_list(),
            profiles["full_history_rows"].to_list(),
        )
    )

In [ ]:
def encode_histories(impression_ids, profiles):
    """Return right-padded chronological histories holding the newest HISTORY_CAP clicks."""
    histories = np.zeros((len(impression_ids), HISTORY_CAP), dtype=np.int64)
    for index, uid in enumerate(impression_ids):
        history = profiles[uid][-HISTORY_CAP:]
        if len(history):
            histories[index, : len(history)] = history
    return histories

In [ ]:
train = load_training_data()
preprocessor = build_preprocessor().fit(train[FEATURES]) if USE_TABULAR else None
train_tabular = tabular_inputs(train)
train_labels = train["clicked"].to_numpy(np.float32)

# Encode each training impression's history once and point every candidate row at it
train_uids = train["impression_uid"].drop_duplicates().tolist()
train_histories = encode_histories(train_uids, load_profiles("train", train_uids))
history_index = (
    train["impression_uid"]
    .map(pd.Series(np.arange(len(train_uids)), index=train_uids))
    .to_numpy(np.int64)
)

candidate_rows = train["news_code"].map(news_to_row)
if candidate_rows.isna().any():
    raise ValueError("A training candidate is missing its article embedding")
train_candidates = candidate_rows.to_numpy(np.int64)
print(
    f"Training rows: {len(train):,}; impressions: {len(train_uids):,}; tabular width: {train_tabular.shape[1]}"
)

# 4. Model

```text
Earlier clicks (chronological) → text embedding → LSTM / GRU → last valid step → user vector
[tabular features ; candidate vector ; user vector] → Dense 256 → Dense 64 → logit
```

## 4.1 Article and History Encoders

In [ ]:
class ArticleEncoder(nn.Module):
    """Map article rows to their frozen 64-dimensional text embeddings (row 0 is padding)."""

    def __init__(self, vectors):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.from_numpy(np.array(vectors, dtype=np.float32, copy=True)),
            freeze=True,
            padding_idx=0,
        )

    def forward(self, article_rows):
        return self.embedding(article_rows)  # [..., 64]


class HistorySequenceEncoder(nn.Module):
    """Run an LSTM or GRU over the chronological click history and keep the last valid step."""

    def __init__(self, article_encoder, rnn_type):
        super().__init__()
        self.article_encoder = article_encoder
        rnn_class = nn.LSTM if rnn_type == "lstm" else nn.GRU
        self.recurrent_layer = rnn_class(EMBEDDING_DIM, EMBEDDING_DIM, batch_first=True)

    def forward(self, history_rows):
        # history_rows: [batch, history], right-padded with 0
        outputs, _ = self.recurrent_layer(self.article_encoder(history_rows))
        lengths = history_rows.ne(0).sum(dim=1)
        last_step = lengths.clamp(min=1) - 1
        user = outputs[
            torch.arange(len(history_rows), device=history_rows.device), last_step
        ]
        return user * lengths.ne(0).unsqueeze(1)  # zero vector for an empty history

## 4.2 Candidate Scorer and Full Reranker

In [ ]:
class CandidateScorer(nn.Module):
    """MLP over [tabular features ; candidate text ; user vector] that returns one logit."""

    def __init__(self, tabular_width, hidden_dims=(256, 64), dropout=0.20):
        super().__init__()
        layers, width = [], tabular_width + 2 * EMBEDDING_DIM
        for hidden in hidden_dims:
            layers.extend([nn.Linear(width, hidden), nn.ReLU(), nn.Dropout(dropout)])
            width = hidden
        layers.append(nn.Linear(width, 1))
        self.layers = nn.Sequential(*layers)

    def forward(self, tabular, candidate_text, user):
        return self.layers(torch.cat([tabular, candidate_text, user], dim=1)).squeeze(
            -1
        )


class SequenceReranker(nn.Module):
    """Article encoder → recurrent user encoder → tabular candidate scorer."""

    def __init__(self, vectors, tabular_width, rnn_type):
        super().__init__()
        self.article_encoder = ArticleEncoder(vectors)
        self.user_encoder = HistorySequenceEncoder(self.article_encoder, rnn_type)
        self.candidate_scorer = CandidateScorer(tabular_width)

    def encode_history(self, history_rows):
        """Encode each impression's history once; candidates of that impression reuse it."""
        return self.user_encoder(history_rows)

    def forward(self, tabular, candidate_rows, user):
        return self.candidate_scorer(
            tabular, self.article_encoder(candidate_rows), user
        )

# 5. Training

## 5.1 LambdaNDCG Loss

For every pair of candidates in an impression where one is clicked and the other is not,
LambdaNDCG applies a RankNet loss $\log(1 + e^{-(s_i - s_j)})$ weighted by $|\Delta \text{nDCG@K}|$,
the change in nDCG@K if the two candidates swapped places. Pairs that cannot change the
top-K therefore contribute little. Padding is masked out of every term.

In [ ]:
def lambda_ndcg_loss(logits, labels, mask, k):
    """RankNet loss weighted by detached |ΔnDCG@k|; inputs are [impressions, candidates]."""
    masked_logits = logits.masked_fill(~mask, float("-inf"))
    ranks = masked_logits.argsort(dim=1, descending=True).argsort(dim=1).add(1).detach()
    gains = (torch.pow(2.0, labels) - 1.0) * mask
    discounts = (1.0 / torch.log2(ranks.float() + 1.0)) * (ranks <= k) * mask

    # Ideal DCG@k for normalisation
    ideal_labels = (
        labels.masked_fill(~mask, float("-inf")).sort(dim=1, descending=True).values
    )
    ideal_mask = torch.isfinite(ideal_labels)
    ideal_labels = ideal_labels.clamp_min(0)
    positions = torch.arange(1, logits.shape[1] + 1, device=logits.device)
    ideal_discounts = (1.0 / torch.log2(positions.float() + 1.0)) * (positions <= k)
    idcg = (
        ((torch.pow(2.0, ideal_labels) - 1.0) * ideal_discounts * ideal_mask)
        .sum(dim=1)
        .clamp_min(1e-12)
    )

    score_diff = logits.unsqueeze(2) - logits.unsqueeze(1)
    delta_ndcg = (
        (gains.unsqueeze(2) - gains.unsqueeze(1))
        * (discounts.unsqueeze(2) - discounts.unsqueeze(1))
    ).abs()
    weights = delta_ndcg / idcg[:, None, None]
    valid_pairs = (
        (labels.unsqueeze(2) > labels.unsqueeze(1))
        & mask.unsqueeze(2)
        & mask.unsqueeze(1)
    )
    weights = weights * valid_pairs
    return (F.softplus(-score_diff) * weights).sum() / weights.sum().clamp_min(1e-12)


def collate_impressions(batch):
    """Pad a batch of impression row groups to one width and return (rows, mask)."""
    groups = [item[0] for item in batch]
    width = max(len(group) for group in groups)
    rows = torch.zeros((len(groups), width), dtype=torch.long)
    mask = torch.zeros((len(groups), width), dtype=torch.bool)
    for index, group in enumerate(groups):
        rows[index, : len(group)] = torch.from_numpy(group)
        mask[index, : len(group)] = True
    return rows, mask

## 5.2 Training Loop

Both objectives score a set of training rows by looking up each row's impression
history, encoding it, and passing it to the scorer with the row's tabular features and
candidate.

In [ ]:
torch.manual_seed(SEED)
model = SequenceReranker(text_vectors, train_tabular.shape[1], RNN_TYPE).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)

tabular_tensor = torch.from_numpy(train_tabular)
candidate_tensor = torch.from_numpy(train_candidates)
label_tensor = torch.from_numpy(train_labels)
history_tensor = torch.from_numpy(train_histories)
history_index_tensor = torch.from_numpy(history_index)


def score_training_rows(rows):
    """Logits for training rows (any shape of row indices, flattened)."""
    rows = rows.reshape(-1)
    users = model.encode_history(history_tensor[history_index_tensor[rows]].to(DEVICE))
    return model(
        tabular_tensor[rows].to(DEVICE), candidate_tensor[rows].to(DEVICE), users
    )


if OBJECTIVE == "bce_bpr":
    positive_rows, negative_rows = load_saved_pairs(train)
    click_rate = train_labels.mean()
    positive_weight = torch.tensor([(1 - click_rate) / click_rate], device=DEVICE)
    row_loader = DataLoader(
        TensorDataset(torch.arange(len(train))),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    pair_loader = DataLoader(
        TensorDataset(torch.from_numpy(positive_rows), torch.from_numpy(negative_rows)),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED + 1),
    )
    print(f"Saved BPR pairs: {len(positive_rows):,}")

    for epoch in range(EPOCHS):
        model.train()
        pair_batches = iter(pair_loader)
        total_loss = 0.0
        for (rows,) in row_loader:
            # Restart the pair stream when it runs out before the candidate stream
            try:
                positive, negative = next(pair_batches)
            except StopIteration:
                pair_batches = iter(pair_loader)
                positive, negative = next(pair_batches)

            optimizer.zero_grad()
            bce = F.binary_cross_entropy_with_logits(
                score_training_rows(rows),
                label_tensor[rows].to(DEVICE),
                pos_weight=positive_weight,
            )
            bpr = -F.logsigmoid(
                score_training_rows(positive) - score_training_rows(negative)
            ).mean()
            loss = (1 - BPR_WEIGHT) * bce + BPR_WEIGHT * bpr
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(
            f"Epoch {epoch + 1}/{EPOCHS}: mean BCE+BPR loss={total_loss / len(row_loader):.5f}"
        )

else:
    # One group of row indices per impression that has both a click and a non-click
    impression_groups = [
        group.astype(np.int64)
        for group in train.groupby("impression_uid", sort=False).indices.values()
    ]
    impression_groups = [
        group
        for group in impression_groups
        if 0 < train_labels[group].sum() < len(group)
    ]
    group_loader = DataLoader(
        [(group,) for group in impression_groups],
        batch_size=BATCH_IMPRESSIONS,
        shuffle=True,
        collate_fn=collate_impressions,
        generator=torch.Generator().manual_seed(SEED),
    )
    print(f"Impression groups with a click and a non-click: {len(impression_groups):,}")

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0
        for rows, mask in group_loader:
            logits = score_training_rows(rows).reshape(rows.shape)
            loss = lambda_ndcg_loss(
                logits, label_tensor[rows].to(DEVICE), mask.to(DEVICE), NDCG_K
            )
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(
            f"Epoch {epoch + 1}/{EPOCHS}: mean LambdaNDCG@{NDCG_K} loss={total_loss / len(group_loader):.5f}"
        )

## 5.3 Save the Model and Preprocessing

Both the weights and the fitted preprocessor are needed for inference.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "state_dict": model.state_dict(),
        "rnn_type": RNN_TYPE,
        "features": FEATURES,
        "seed": SEED,
    },
    OUTPUT_DIR / "model.pt",
)
if USE_TABULAR:
    joblib.dump(preprocessor, OUTPUT_DIR / "preprocessor.joblib")
(OUTPUT_DIR / "model_metadata.json").write_text(
    json.dumps(
        {
            "model_type": "sequence_reranker",
            "use_tabular": USE_TABULAR,
            "rnn_type": RNN_TYPE,
            "objective": OBJECTIVE,
            "bpr_weight": BPR_WEIGHT if OBJECTIVE == "bce_bpr" else None,
            "ndcg_k": NDCG_K if OBJECTIVE == "lambda_ndcg" else None,
            "history_cap": HISTORY_CAP,
            "embedding": "text64",
            "epochs": EPOCHS,
            "train_rows": len(train),
            "seed": SEED,
        },
        indent=2,
    )
    + "\n"
)

# 6. Validation and Test Evaluation

The history of each impression is encoded once and shared by all of its candidates.
Compare `RNN_TYPE` and `OBJECTIVE` variants on validation only; test is for reporting.

In [ ]:
def sequence_predictor(profiles):
    """Wrap the trained model and a split's histories as a predict(frame) function."""

    def predict(frame):
        uids = frame["impression_uid"].drop_duplicates().tolist()
        histories = torch.from_numpy(encode_histories(uids, profiles))
        model.eval()
        with torch.no_grad():
            users = torch.cat(
                [
                    model.encode_history(
                        histories[start : start + BATCH_SIZE].to(DEVICE)
                    ).cpu()
                    for start in range(0, len(uids), BATCH_SIZE)
                ]
            ).numpy()

        user_rows = (
            frame["impression_uid"]
            .map(pd.Series(np.arange(len(uids)), index=uids))
            .to_numpy()
        )
        tabular = tabular_inputs(frame)
        candidates = frame["news_code"].map(news_to_row).to_numpy(np.int64)

        scores = []
        with torch.no_grad():
            for start in range(0, len(frame), BATCH_SIZE):
                stop = start + BATCH_SIZE
                logits = model(
                    torch.from_numpy(tabular[start:stop]).to(DEVICE),
                    torch.from_numpy(candidates[start:stop]).to(DEVICE),
                    torch.from_numpy(users[user_rows[start:stop]]).to(DEVICE),
                )
                scores.append(torch.sigmoid(logits).cpu().numpy())
        return np.concatenate(scores)

    return predict


split_results = []
for split in ("validation", "test"):
    result, _ = score_and_evaluate(
        sequence_predictor(load_profiles(split)),
        split=split,
        feature_run=RUN,
        output_dir=OUTPUT_DIR,
        model_name=EXPERIMENT_NAME,
        columns=TABULAR_COLUMNS,
        results_dir=RESULTS,
    )
    split_results.append(result)

display(
    pl.concat(split_results).select(
        "model", "split", "mrr", "ndcg_at_5", "ndcg_at_10", "recall_at_10"
    )
)

# 7. Branch Ablation: With and Without Tabular Features

Run the notebook once with `USE_TABULAR = True` and once with `False`. This cell then
compares the best validation trial of each run: same `RNN_TYPE` and `OBJECTIVE`, so the gap is what the tabular features add.

In [ ]:
branch_results = []
for branches in ("text", "tabular_text"):
    path = (
        RESULTS
        / f"reranking_{sequence_name(rnn_type=RNN_TYPE, objective=objective_name, history_cap=HISTORY_CAP, hidden_dims=HIDDEN_DIMS, dropout=DROPOUT, epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED, branches=branches)}_validation.csv"
    )
    if path.exists():
        branch_results.append(
            pl.read_csv(path)
            .sort(SELECTION_METRIC, descending=True)
            .head(1)
            .with_columns(pl.lit(branches).alias("branches"))
        )
if branch_results:
    display(
        compare_results(branch_results, sort_by=SELECTION_METRIC).select(
            "branches", "model", "mrr", "ndcg_at_5", "ndcg_at_10", "recall_at_10"
        )
    )
else:
    print("Run the notebook with USE_TABULAR = True and False to fill this comparison.")

# 8. Recommendations for a User

`rank_logged_slate` reorders the articles shown in a user's latest validation
impression using this notebook's saved scores. It does not retrieve new articles
from the catalogue and never reads click labels.

In [ ]:
example_user = int(validation_preview["user_code"].iloc[0])
rank_logged_slate(
    example_user,
    score_file=OUTPUT_DIR / "validation_scores.parquet",
    feature_run=RUN,
    top_n=5,
)